# Flipkart Gridlock 2.0 — Bengaluru Traffic Demand Prediction
**Objective:** Maximize R² (Score = max(0, 100 × R²))

**Strategy:** CatBoostRegressor with geohash×timestamp target encoding as the dominant signal. Validated using a timestamp-aligned holdout that mirrors the test set distribution.

## 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostRegressor
from sklearn.metrics import r2_score
import lightgbm as lgb

# Reproducibility
SEED = 42
np.random.seed(SEED)

print('Libraries loaded successfully.')

Libraries loaded successfully.


## 2. Load Data

In [2]:
# ── Adjust paths if needed ───────────────────────────────────────────────────
TRAIN_PATH  = 'train.csv'
TEST_PATH   = 'test.csv'
SUB_PATH    = 'sample_submission.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sub   = pd.read_csv(SUB_PATH)

print(f'Train shape : {train.shape}')
print(f'Test  shape : {test.shape}')
print(f'Sample sub  : {sub.shape}')
print('\nTrain columns:', train.columns.tolist())
print('\nTrain dtypes:\n', train.dtypes)
print('\nMissing values (train):\n', train.isnull().sum())
print('\nMissing values (test):\n',  test.isnull().sum())

Train shape : (77299, 11)
Test  shape : (41778, 10)
Sample sub  : (5, 2)

Train columns: ['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather']

Train dtypes:
 Index              int64
geohash              str
day                int64
timestamp            str
demand           float64
RoadType             str
NumberofLanes      int64
LargeVehicles        str
Landmarks            str
Temperature      float64
Weather              str
dtype: object

Missing values (train):
 Index               0
geohash             0
day                 0
timestamp           0
demand              0
RoadType          600
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      2495
Weather           797
dtype: int64

Missing values (test):
 Index               0
geohash             0
day                 0
timestamp           0
RoadType          324
NumberofLanes       0
LargeVehicles       0
Landmarks 

## 3. EDA Snapshot

In [3]:
print('=== TARGET DISTRIBUTION ===')
print(train['demand'].describe())

print('\n=== KEY CARDINALITIES ===')
for col in ['geohash','day','timestamp','RoadType','NumberofLanes','LargeVehicles','Landmarks','Weather']:
    print(f'  {col}: {train[col].nunique()} unique')

# Critical insight: test is day=49 only; train day 49 covers only first ~2h of night
print('\n=== DAY DISTRIBUTION ===')
print(train['day'].value_counts())
print('Test day:', test['day'].unique())

print('\n=== DEMAND BY ROADTYPE ===')
print(train.groupby('RoadType')['demand'].agg(['mean','std','count']).round(4))

# Test timestamp window: 2:15 – 13:45  (47 of 96 possible 15-min slots)
def ts_to_min(t):
    h, m = t.split(':')
    return int(h)*60 + int(m)

test_ts_set = set(test['timestamp'].unique())
print(f'\nTest timestamps ({len(test_ts_set)}): {sorted(test_ts_set, key=ts_to_min)[:5]} … {sorted(test_ts_set, key=ts_to_min)[-3:]}')

# Geohash coverage
train_geo = set(train['geohash'])
test_geo  = set(test['geohash'])
print(f'\nGeohash coverage: {len(test_geo & train_geo)}/{len(test_geo)} test geohashes seen in train')

# geo×ts coverage
train_pairs = set(zip(train['geohash'], train['timestamp']))
test_pairs  = set(zip(test['geohash'],  test['timestamp']))
print(f'geo×timestamp coverage: {len(test_pairs & train_pairs)}/{len(test_pairs)} pairs seen in train')

=== TARGET DISTRIBUTION ===
count    7.729900e+04
mean     9.394238e-02
std      1.421905e-01
min      6.245650e-07
25%      1.822723e-02
50%      4.775994e-02
75%      1.085951e-01
max      1.000000e+00
Name: demand, dtype: float64

=== KEY CARDINALITIES ===
  geohash: 1249 unique
  day: 2 unique
  timestamp: 96 unique
  RoadType: 3 unique
  NumberofLanes: 5 unique
  LargeVehicles: 2 unique
  Landmarks: 2 unique
  Weather: 4 unique

=== DAY DISTRIBUTION ===
day
48    69427
49     7872
Name: count, dtype: int64
Test day: [49]

=== DEMAND BY ROADTYPE ===
               mean     std  count
RoadType                          
Highway      0.6108  0.2294   3560
Residential  0.0572  0.0521  69230
Street       0.2732  0.0367   3909

Test timestamps (47): ['2:15', '2:30', '2:45', '3:0', '3:15'] … ['13:15', '13:30', '13:45']

Geohash coverage: 1180/1190 test geohashes seen in train
geo×timestamp coverage: 37136/41778 pairs seen in train


## 4. Validation Strategy

**Design rationale:**
- Test set = day 49, timestamps 2:15–13:45 only.
- Train day 49 covers only timestamps 0:00–2:00 → **not representative** of test.
- Best proxy: day 48 rows **with test-matching timestamps** as the holdout.
- Train fold: day 48 rows outside that window + all of day 49.
- Target encodings computed strictly from the train fold to avoid leakage.

In [4]:
def ts_to_min(t):
    h, m = t.split(':')
    return int(h)*60 + int(m)

TEST_TS_SET = set(test['timestamp'].unique())   # 47 timestamps matching test window

train48 = train[train['day'] == 48].copy()
train49 = train[train['day'] == 49].copy()

val_df = train48[train48['timestamp'].isin(TEST_TS_SET)].copy().reset_index(drop=True)
tr_df  = pd.concat([
    train48[~train48['timestamp'].isin(TEST_TS_SET)],
    train49
], ignore_index=True)

print(f'Train fold : {len(tr_df):,} rows')
print(f'Val fold   : {len(val_df):,} rows  (mirrors test timestamp window)')
print(f'Val geo coverage: {val_df["geohash"].nunique()} / {train["geohash"].nunique()} geohashes')

Train fold : 35,448 rows
Val fold   : 41,851 rows  (mirrors test timestamp window)
Val geo coverage: 1224 / 1249 geohashes


## 5. Feature Engineering

In [5]:
def build_features(df: pd.DataFrame, ref_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build all features for df using ref_df as the encoding source.
    For validation: ref_df = tr_df
    For test:       ref_df = full train
    This prevents target leakage.
    """
    df = df.copy()
    global_mean = ref_df['demand'].mean()

    # Missing indicators must be captured before imputation.
    df['temp_missing'] = df['Temperature'].isna().astype(int)
    df['weather_missing'] = df['Weather'].isna().astype(int)
    df['rt_missing'] = df['RoadType'].isna().astype(int)

    # Timestamp features
    df['ts_min'] = df['timestamp'].apply(ts_to_min)
    df['hour'] = df['ts_min'] // 60
    df['minute_slot'] = (df['ts_min'] % 60) // 15
    df['time_slot'] = df['ts_min'] // 15
    df['is_rush_am'] = ((df['hour'] >= 7) & (df['hour'] <= 9)).astype(int)
    df['is_rush_pm'] = ((df['hour'] >= 17) & (df['hour'] <= 20)).astype(int)
    df['is_night'] = ((df['hour'] >= 23) | (df['hour'] <= 5)).astype(int)
    df['sin_hour'] = np.sin(2 * np.pi * df['ts_min'] / 1440)
    df['cos_hour'] = np.cos(2 * np.pi * df['ts_min'] / 1440)
    df['sin_time_slot'] = np.sin(2 * np.pi * df['time_slot'] / 96)
    df['cos_time_slot'] = np.cos(2 * np.pi * df['time_slot'] / 96)

    # Geohash prefix features
    df['geo4'] = df['geohash'].str[:4]
    df['geo5'] = df['geohash'].str[:5]

    # Road type imputation + ordinal
    rt_order = {'Residential': 0, 'Street': 1, 'Highway': 2}
    geo_rt_mode = (
        ref_df.groupby('geohash')['RoadType']
              .agg(lambda x: x.dropna().mode()[0] if not x.dropna().empty else 'Residential')
              .to_dict()
    )
    df['road_type_filled'] = df['RoadType'].copy()
    rt_na = df['road_type_filled'].isna()
    df.loc[rt_na, 'road_type_filled'] = df.loc[rt_na, 'geohash'].map(geo_rt_mode)
    df['road_type_filled'] = df['road_type_filled'].fillna('Residential')
    df['road_type_ord'] = df['road_type_filled'].map(rt_order).fillna(0).astype(int)

    # Binary flags & interactions
    df['large_veh_bin'] = (df['LargeVehicles'] == 'Allowed').astype(int)
    df['landmark_bin'] = (df['Landmarks'] == 'Yes').astype(int)
    df['lanes_x_road'] = df['NumberofLanes'] * df['road_type_ord']

    # Temperature imputation
    weather_temp_mean = ref_df.groupby('Weather')['Temperature'].mean().to_dict()
    global_temp = ref_df['Temperature'].mean()
    df['temp_filled'] = df['Temperature'].copy()
    t_na = df['temp_filled'].isna()
    df.loc[t_na, 'temp_filled'] = df.loc[t_na, 'Weather'].map(weather_temp_mean)
    df['temp_filled'] = df['temp_filled'].fillna(global_temp)
    df['weather_filled'] = df['Weather'].fillna('Sunny')

    # Target encodings computed from ref_df only
    ref_enc = ref_df.copy()
    ref_enc['geo4'] = ref_enc['geohash'].str[:4]
    ref_enc['geo5'] = ref_enc['geohash'].str[:5]
    ref_enc['hour'] = ref_enc['timestamp'].apply(ts_to_min) // 60

    geo_mean = ref_enc.groupby('geohash')['demand'].mean().to_dict()
    geo4_mean = ref_enc.groupby('geo4')['demand'].mean().to_dict()
    geo5_mean = ref_enc.groupby('geo5')['demand'].mean().to_dict()
    geo_ts_dict = ref_enc.groupby(['geohash', 'timestamp'])['demand'].mean().to_dict()
    geo_hour_dict = ref_enc.groupby(['geohash', 'hour'])['demand'].mean().to_dict()
    rt_ts_dict = ref_enc.groupby(['RoadType', 'timestamp'])['demand'].mean().to_dict()
    rt_hour_dict = ref_enc.groupby(['RoadType', 'hour'])['demand'].mean().to_dict()
    ts_mean_dict = ref_enc.groupby('timestamp')['demand'].mean().to_dict()
    geo_std_dict = ref_enc.groupby('geohash')['demand'].std().fillna(0).to_dict()
    geo_count_dict = ref_enc.groupby('geohash')['demand'].count().to_dict()

    # Count and sum statistics for confidence and smoothed encodings.
    geo_ts_count_dict = ref_enc.groupby(['geohash', 'timestamp'])['demand'].count().to_dict()
    geo_hour_count_dict = ref_enc.groupby(['geohash', 'hour'])['demand'].count().to_dict()
    rt_ts_count_dict = ref_enc.groupby(['RoadType', 'timestamp'])['demand'].count().to_dict()
    geo_ts_sum_dict = ref_enc.groupby(['geohash', 'timestamp'])['demand'].sum().to_dict()
    geo_hour_sum_dict = ref_enc.groupby(['geohash', 'hour'])['demand'].sum().to_dict()
    rt_ts_sum_dict = ref_enc.groupby(['RoadType', 'timestamp'])['demand'].sum().to_dict()

    # Frequency encodings from ref_df only.
    geo_freq = ref_enc['geohash'].value_counts(normalize=True).to_dict()
    geo4_freq = ref_enc['geo4'].value_counts(normalize=True).to_dict()
    geo5_freq = ref_enc['geo5'].value_counts(normalize=True).to_dict()

    df['geo_mean'] = df['geohash'].map(geo_mean).fillna(global_mean)
    df['geo4_mean'] = df['geo4'].map(geo4_mean).fillna(global_mean)
    df['geo5_mean'] = df['geo5'].map(geo5_mean).fillna(df['geo4_mean']).fillna(global_mean)

    # geo x timestamp, using the existing leakage-safe methodology.
    df['geo_ts_mean'] = [geo_ts_dict.get((g, ts), np.nan)
                         for g, ts in zip(df['geohash'], df['timestamp'])]
    df['geo_ts_mean'] = df['geo_ts_mean'].fillna(df['geo_mean'])

    # geo x hour, fallback when exact timestamp unseen.
    df['geo_hour_mean'] = [geo_hour_dict.get((g, h), np.nan)
                           for g, h in zip(df['geohash'], df['hour'])]
    df['geo_hour_mean'] = df['geo_hour_mean'].fillna(df['geo_mean'])

    df['rt_ts_mean'] = [rt_ts_dict.get((rt, ts), np.nan)
                        for rt, ts in zip(df['road_type_filled'], df['timestamp'])]
    df['rt_ts_mean'] = df['rt_ts_mean'].fillna(global_mean)

    df['rt_hour_mean'] = [rt_hour_dict.get((rt, h), np.nan)
                          for rt, h in zip(df['road_type_filled'], df['hour'])]
    df['rt_hour_mean'] = df['rt_hour_mean'].fillna(global_mean)

    df['ts_mean'] = df['timestamp'].map(ts_mean_dict).fillna(global_mean)
    df['geo_std'] = df['geohash'].map(geo_std_dict).fillna(0)
    df['geo_count'] = df['geohash'].map(geo_count_dict).fillna(0)
    df['geo_freq'] = df['geohash'].map(geo_freq).fillna(0)
    df['geo4_freq'] = df['geo4'].map(geo4_freq).fillna(0)
    df['geo5_freq'] = df['geo5'].map(geo5_freq).fillna(0)

    df['geo_ts_count'] = [geo_ts_count_dict.get((g, ts), 0)
                          for g, ts in zip(df['geohash'], df['timestamp'])]
    df['geo_hour_count'] = [geo_hour_count_dict.get((g, h), 0)
                            for g, h in zip(df['geohash'], df['hour'])]
    df['rt_ts_count'] = [rt_ts_count_dict.get((rt, ts), 0)
                         for rt, ts in zip(df['road_type_filled'], df['timestamp'])]

    df['geo_ts_seen'] = (df['geo_ts_count'] > 0).astype(int)
    df['geo_hour_seen'] = (df['geo_hour_count'] > 0).astype(int)
    df['rt_ts_seen'] = (df['rt_ts_count'] > 0).astype(int)

    for smooth_m in [5, 10, 20, 50]:
        geo_ts_sum = np.array([geo_ts_sum_dict.get((g, ts), 0.0)
                               for g, ts in zip(df['geohash'], df['timestamp'])])
        geo_hour_sum = np.array([geo_hour_sum_dict.get((g, h), 0.0)
                                 for g, h in zip(df['geohash'], df['hour'])])
        rt_ts_sum = np.array([rt_ts_sum_dict.get((rt, ts), 0.0)
                              for rt, ts in zip(df['road_type_filled'], df['timestamp'])])

        df[f'geo_ts_smooth_m{smooth_m}'] = (
            geo_ts_sum + global_mean * smooth_m
        ) / (df['geo_ts_count'].to_numpy() + smooth_m)
        df[f'geo_hour_smooth_m{smooth_m}'] = (
            geo_hour_sum + global_mean * smooth_m
        ) / (df['geo_hour_count'].to_numpy() + smooth_m)
        df[f'rt_ts_smooth_m{smooth_m}'] = (
            rt_ts_sum + global_mean * smooth_m
        ) / (df['rt_ts_count'].to_numpy() + smooth_m)

    df['geo_ts_confidence'] = df['geo_ts_mean'] * np.log1p(df['geo_ts_count'])
    df['geo_hour_confidence'] = df['geo_hour_mean'] * np.log1p(df['geo_hour_count'])
    df['rt_ts_confidence'] = df['rt_ts_mean'] * np.log1p(df['rt_ts_count'])

    # Delta: how much demand deviates at this time vs the geohash average.
    df['geo_ts_delta'] = df['geo_ts_mean'] - df['geo_mean']

    return df


print('Feature engineering function defined.')

Feature engineering function defined.


## 6. Train CatBoost — Validation Fold

In [6]:
print('Building train fold features...')
tr_fe = build_features(tr_df, tr_df)
print('Building val fold features...')
val_fe = build_features(val_df, tr_df)
print('Done. Shape:', tr_fe.shape)

BASE_FEATURES = [
    # Time
    'ts_min', 'hour', 'minute_slot', 'is_rush_am', 'is_rush_pm', 'is_night',
    'sin_hour', 'cos_hour',
    # Road / infrastructure
    'NumberofLanes', 'large_veh_bin', 'landmark_bin', 'lanes_x_road',
    'temp_filled', 'road_type_ord',
    # Target encodings
    'geo_mean', 'geo4_mean', 'geo_ts_mean', 'geo_hour_mean',
    'rt_ts_mean', 'rt_hour_mean', 'ts_mean', 'geo_std', 'geo_count',
    'geo_ts_delta',
    # Helpful v2 features retained
    'geo_freq', 'geo4_freq', 'geo5_freq',
    'temp_missing', 'weather_missing', 'rt_missing',
    # Categoricals handled natively by CatBoost
    'geohash', 'geo4', 'road_type_filled', 'weather_filled'
]

BASE_CAT_FEATURES = ['geohash', 'geo4', 'road_type_filled', 'weather_filled']

COUNT_FEATURES = ['geo_ts_count', 'geo_hour_count', 'rt_ts_count']  # geo_count already exists in BASE_FEATURES
SEEN_FEATURES = ['geo_ts_seen', 'geo_hour_seen', 'rt_ts_seen']
SMOOTH_MS = [5, 10, 20, 50]
CONFIDENCE_FEATURES = ['geo_ts_confidence', 'geo_hour_confidence', 'rt_ts_confidence']


def add_unique(base, additions):
    result = list(base)
    for item in additions:
        if item not in result:
            result.append(item)
    return result


active_features = BASE_FEATURES.copy()
active_cat_features = BASE_CAT_FEATURES.copy()

y_tr = tr_fe['demand']
y_val = val_fe['demand']

print(f'Training on {len(tr_fe):,} rows, validating on {len(val_fe):,} rows.')
print(f'Current v2 feature set: {len(active_features)} ({len(active_cat_features)} categorical)')

Building train fold features...
Building val fold features...
Done. Shape: (35448, 69)
Training on 35,448 rows, validating on 41,851 rows.
Current v2 feature set: 34 (4 categorical)


In [7]:
CAT_BASE_PARAMS = dict(
    iterations=5000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=7,
    min_data_in_leaf=20,
    bagging_temperature=1.0,
    random_strength=1.0,
    eval_metric='R2',
    loss_function='RMSE',
    random_seed=SEED,
    early_stopping_rounds=200,
)


def train_catboost_eval(features, cat_features, param_overrides=None, verbose=False):
    params = CAT_BASE_PARAMS.copy()
    if param_overrides:
        params.update(param_overrides)

    model = CatBoostRegressor(
        **params,
        cat_features=[c for c in cat_features if c in features],
        verbose=verbose,
    )
    model.fit(
        tr_fe[features], y_tr,
        eval_set=(val_fe[features], y_val),
        use_best_model=True,
    )
    preds = model.predict(val_fe[features])
    score = r2_score(y_val, preds)
    return score, model, preds


# Phase 1 diagnostics
raw_diagnostics = {
    'geo_ts_mean': r2_score(y_val, val_fe['geo_ts_mean']),
    'geo_hour_mean': r2_score(y_val, val_fe['geo_hour_mean']),
    'geo_mean': r2_score(y_val, val_fe['geo_mean']),
    'rt_ts_mean': r2_score(y_val, val_fe['rt_ts_mean']),
    'ts_mean': r2_score(y_val, val_fe['ts_mean']),
}
coverage = {
    'geo_ts_hit_rate': val_fe['geo_ts_seen'].mean(),
    'geo_ts_hits': int(val_fe['geo_ts_seen'].sum()),
    'geo_hour_hit_rate': val_fe['geo_hour_seen'].mean(),
    'geo_hour_hits': int(val_fe['geo_hour_seen'].sum()),
    'validation_rows': len(val_fe),
}

print('=== PHASE 1 DIAGNOSTICS ===')
for feature_name, score in raw_diagnostics.items():
    print(f'{feature_name:<14} raw R2: {score:.6f}')
print(f"geo x timestamp coverage: {coverage['geo_ts_hits']:,}/{coverage['validation_rows']:,} = {coverage['geo_ts_hit_rate']:.3%}")
print(f"geo x hour coverage     : {coverage['geo_hour_hits']:,}/{coverage['validation_rows']:,} = {coverage['geo_hour_hit_rate']:.3%}")
print('Interpretation: geo_ts_mean has no exact validation support and falls back to geo_mean; geo_hour_mean has limited support and is the strongest raw encoding.')

print()
print('Training current v2 CatBoost baseline...')
current_r2, current_model, current_preds = train_catboost_eval(active_features, active_cat_features)
print(f'Current feature-set CatBoost R2: {current_r2:.6f}')

feature_eval_rows = []

def evaluate_group(group_name, candidate_features, candidate_cat_features=None):
    global active_features, active_cat_features, current_r2, current_model, current_preds
    if candidate_cat_features is None:
        candidate_cat_features = active_cat_features

    before = current_r2
    after, candidate_model, candidate_preds = train_catboost_eval(candidate_features, candidate_cat_features)
    gain = after - before
    keep = gain > 0

    if keep:
        active_features = candidate_features
        active_cat_features = candidate_cat_features
        current_r2 = after
        current_model = candidate_model
        current_preds = candidate_preds

    feature_eval_rows.append({
        'Experiment': group_name,
        'Validation R2 Before': before,
        'Validation R2 After': after,
        'Gain': gain,
        'Kept': keep,
    })

    print()
    print(f'{group_name}')
    print(f'Validation R2 Before: {before:.6f}')
    print(f'Validation R2 After : {after:.6f}')
    print(f'Gain                : {gain:+.6f}')
    print('Decision            :', 'kept' if keep else 'removed from final feature set')
    return after, gain, keep


# Phase 2: count features
candidate_features = add_unique(active_features, COUNT_FEATURES)
evaluate_group('count_features', candidate_features)
print('Note: geo_count was already present in the current v2 feature set; new count candidates are geo_ts_count, geo_hour_count, rt_ts_count.')

# Phase 3: seen/unseen indicators
candidate_features = add_unique(active_features, SEEN_FEATURES)
evaluate_group('seen_unseen_features', candidate_features)

# Phase 4: smoothed target encodings
smooth_rows = []
smooth_baseline = current_r2
best_smooth = {'M': None, 'score': -np.inf, 'features': None}
for smooth_m in SMOOTH_MS:
    smooth_features = [f'geo_ts_smooth_m{smooth_m}', f'geo_hour_smooth_m{smooth_m}', f'rt_ts_smooth_m{smooth_m}']
    candidate_features = add_unique(active_features, smooth_features)
    score, _, _ = train_catboost_eval(candidate_features, active_cat_features)
    smooth_rows.append({'M': smooth_m, 'Validation R2': score, 'Gain vs Before': score - smooth_baseline})
    print(f'Smoothing M={smooth_m:<2} -> R2={score:.6f} gain={score - smooth_baseline:+.6f}')
    if score > best_smooth['score']:
        best_smooth = {'M': smooth_m, 'score': score, 'features': smooth_features}

if best_smooth['score'] > current_r2:
    before = current_r2
    active_features = add_unique(active_features, best_smooth['features'])
    current_r2, current_model, current_preds = train_catboost_eval(active_features, active_cat_features)
    feature_eval_rows.append({
        'Experiment': f"smoothed_encodings_m{best_smooth['M']}",
        'Validation R2 Before': before,
        'Validation R2 After': current_r2,
        'Gain': current_r2 - before,
        'Kept': True,
    })
    print(f"Best smoothing M={best_smooth['M']} kept with R2={current_r2:.6f}")
else:
    feature_eval_rows.append({
        'Experiment': 'smoothed_encodings',
        'Validation R2 Before': current_r2,
        'Validation R2 After': best_smooth['score'],
        'Gain': best_smooth['score'] - current_r2,
        'Kept': False,
    })
    print(f"Best smoothing M={best_smooth['M']} did not improve and was removed.")

smooth_df = pd.DataFrame(smooth_rows)

# Phase 5: confidence-weighted features, evaluated one at a time.
for confidence_feature in CONFIDENCE_FEATURES:
    candidate_features = add_unique(active_features, [confidence_feature])
    evaluate_group(confidence_feature, candidate_features)

feature_eval_df = pd.DataFrame(feature_eval_rows)
ranked_improvements = feature_eval_df.sort_values('Gain', ascending=False).reset_index(drop=True)

print()
print('Selected v3 feature additions:', feature_eval_df.loc[feature_eval_df['Kept'], 'Experiment'].tolist())
print(f'Selected feature count: {len(active_features)} ({len(active_cat_features)} categorical)')

# Phase 6: micro hyperparameter search on selected features.
print()
print('=== PHASE 6 MICRO HYPERPARAMETER SEARCH ===')
tuning_rows = []
best_cat_r2 = current_r2
best_cat_model = current_model
best_cat_preds = current_preds
best_cat_params = {'depth': 6, 'l2_leaf_reg': 7, 'learning_rate': 0.05}

for learning_rate in [0.03, 0.05]:
    for depth in [5, 6, 7]:
        for l2_leaf_reg in [5, 7, 10]:
            overrides = {'learning_rate': learning_rate, 'depth': depth, 'l2_leaf_reg': l2_leaf_reg}
            if (learning_rate == 0.05 and depth == 6 and l2_leaf_reg == 7):
                score, model, preds = current_r2, current_model, current_preds
            else:
                score, model, preds = train_catboost_eval(active_features, active_cat_features, overrides)

            tuning_rows.append({
                'learning_rate': learning_rate,
                'depth': depth,
                'l2_leaf_reg': l2_leaf_reg,
                'Validation R2': score,
            })
            print(f'lr={learning_rate:<4} depth={depth} l2={l2_leaf_reg:<2} -> R2={score:.6f}')

            if score > best_cat_r2:
                best_cat_r2 = score
                best_cat_model = model
                best_cat_preds = preds
                best_cat_params = overrides

cat_tuning_df = pd.DataFrame(tuning_rows).sort_values('Validation R2', ascending=False).reset_index(drop=True)
print()
print('Best CatBoost params:', best_cat_params)
print(f'Best CatBoost R2: {best_cat_r2:.6f}')

# Phase 7: CatBoost ensemble.
print()
print('=== PHASE 7 CATBOOST ENSEMBLE ===')
ensemble_specs = {
    'A': {'learning_rate': 0.05, 'depth': 6, 'l2_leaf_reg': 5},
    'B': {'learning_rate': 0.05, 'depth': 6, 'l2_leaf_reg': 7},
    'C': {'learning_rate': 0.05, 'depth': 6, 'l2_leaf_reg': 10},
}
ensemble_models = {}
ensemble_preds = {}
ensemble_single_rows = []
for name, params in ensemble_specs.items():
    score, model, preds = train_catboost_eval(active_features, active_cat_features, params)
    ensemble_models[name] = model
    ensemble_preds[name] = preds
    ensemble_single_rows.append({'Model': name, **params, 'Validation R2': score})
    print(f"Model {name} {params} -> R2={score:.6f}")

blend_rows = []
weight_grid = np.round(np.arange(0.05, 1.00, 0.05), 2)
for combo in [('A', 'B'), ('A', 'C'), ('B', 'C')]:
    best_combo = {'score': -np.inf, 'weights': None}
    for w in weight_grid:
        preds = w * ensemble_preds[combo[0]] + (1 - w) * ensemble_preds[combo[1]]
        score = r2_score(y_val, preds)
        if score > best_combo['score']:
            best_combo = {'score': score, 'weights': {combo[0]: float(w), combo[1]: float(1 - w)}}
    blend_rows.append({'Blend': '+'.join(combo), 'Weights': best_combo['weights'], 'Validation R2': best_combo['score']})
    print(f"Blend {'+'.join(combo)} best weights {best_combo['weights']} -> R2={best_combo['score']:.6f}")

best_combo = {'score': -np.inf, 'weights': None}
for w_a in weight_grid:
    for w_b in weight_grid:
        w_c = round(1 - w_a - w_b, 2)
        if w_c < 0.05:
            continue
        preds = w_a * ensemble_preds['A'] + w_b * ensemble_preds['B'] + w_c * ensemble_preds['C']
        score = r2_score(y_val, preds)
        if score > best_combo['score']:
            best_combo = {'score': score, 'weights': {'A': float(w_a), 'B': float(w_b), 'C': float(w_c)}}
blend_rows.append({'Blend': 'A+B+C', 'Weights': best_combo['weights'], 'Validation R2': best_combo['score']})
print(f"Blend A+B+C best weights {best_combo['weights']} -> R2={best_combo['score']:.6f}")

ensemble_single_df = pd.DataFrame(ensemble_single_rows).sort_values('Validation R2', ascending=False).reset_index(drop=True)
ensemble_df = pd.DataFrame(blend_rows).sort_values('Validation R2', ascending=False).reset_index(drop=True)
best_ensemble = ensemble_df.iloc[0].to_dict()
best_ensemble_r2 = float(best_ensemble['Validation R2'])

if best_ensemble_r2 > best_cat_r2:
    final_strategy = 'ensemble'
    best_validation_r2 = best_ensemble_r2
else:
    final_strategy = 'single_catboost'
    best_validation_r2 = best_cat_r2

print()
print('Best ensemble:', best_ensemble)
print(f'Best validation R2 selected for final output: {best_validation_r2:.6f} ({final_strategy})')

final_features = active_features.copy()
final_cat_features = active_cat_features.copy()

=== PHASE 1 DIAGNOSTICS ===
geo_ts_mean    raw R2: 0.569558
geo_hour_mean  raw R2: 0.574106
geo_mean       raw R2: 0.569558
rt_ts_mean     raw R2: -0.028544
ts_mean        raw R2: -0.028544
geo x timestamp coverage: 0/41,851 = 0.000%
geo x hour coverage     : 2,556/41,851 = 6.107%
Interpretation: geo_ts_mean has no exact validation support and falls back to geo_mean; geo_hour_mean has limited support and is the strongest raw encoding.

Training current v2 CatBoost baseline...
Current feature-set CatBoost R2: 0.664431

count_features
Validation R2 Before: 0.664431
Validation R2 After : 0.574533
Gain                : -0.089898
Decision            : removed from final feature set
Note: geo_count was already present in the current v2 feature set; new count candidates are geo_ts_count, geo_hour_count, rt_ts_count.

seen_unseen_features
Validation R2 Before: 0.664431
Validation R2 After : 0.664431
Gain                : +0.000000
Decision            : removed from final feature set
Smoothing 

## 7. Validation Score & Feature Importance

In [8]:
# Phase 8: feature importance analysis.
if final_strategy == 'ensemble':
    final_weights = best_ensemble['Weights']
    importance_parts = []
    for model_name, weight in final_weights.items():
        importance_parts.append(
            pd.Series(ensemble_models[model_name].get_feature_importance(), index=final_features) * weight
        )
    fi = sum(importance_parts).sort_values(ascending=False)
else:
    fi = pd.Series(
        best_cat_model.get_feature_importance(),
        index=final_features,
    ).sort_values(ascending=False)

print('=== FEATURE IMPORTANCE (top 30, final validation strategy) ===')
print(fi.head(30).round(4).to_string())

inspect_features = [
    'geo_ts_mean', 'geo_ts_count', 'geo_ts_seen',
    'geo_hour_mean', 'geo_hour_count', 'geo_hour_seen',
    'rt_ts_mean', 'rt_ts_count', 'rt_ts_seen',
    'geo_ts_smooth_m5', 'geo_ts_smooth_m10', 'geo_ts_smooth_m20', 'geo_ts_smooth_m50',
    'geo_hour_smooth_m5', 'geo_hour_smooth_m10', 'geo_hour_smooth_m20', 'geo_hour_smooth_m50',
    'geo_freq', 'geo4_freq', 'geo5_freq',
]
importance_review = pd.DataFrame({
    'Feature': inspect_features,
    'In Final Features': [f in final_features for f in inspect_features],
    'Importance': [float(fi.get(f, 0.0)) for f in inspect_features],
}).sort_values(['In Final Features', 'Importance'], ascending=[False, False])

print()
print('=== TARGETED IMPORTANCE REVIEW ===')
print(importance_review.to_string(index=False))

print()
print('=== FEATURE EXPERIMENT RESULTS ===')
print(feature_eval_df.assign(
    **{
        'Validation R2 Before': feature_eval_df['Validation R2 Before'].round(6),
        'Validation R2 After': feature_eval_df['Validation R2 After'].round(6),
        'Gain': feature_eval_df['Gain'].round(6),
    }
).to_string(index=False))

print()
print('=== SMOOTHING RESULTS ===')
print(smooth_df.assign(
    **{
        'Validation R2': smooth_df['Validation R2'].round(6),
        'Gain vs Before': smooth_df['Gain vs Before'].round(6),
    }
).to_string(index=False))

print()
print('=== HYPERPARAMETER RESULTS ===')
print(cat_tuning_df.assign(**{'Validation R2': cat_tuning_df['Validation R2'].round(6)}).to_string(index=False))

print()
print('=== ENSEMBLE SINGLE MODELS ===')
print(ensemble_single_df.assign(**{'Validation R2': ensemble_single_df['Validation R2'].round(6)}).to_string(index=False))

print()
print('=== ENSEMBLE BLENDS ===')
print(ensemble_df.assign(**{'Validation R2': ensemble_df['Validation R2'].round(6)}).to_string(index=False))

print()
print('=== FINAL FEATURE LIST ===')
print(pd.Series(final_features).to_string(index=False))

=== FEATURE IMPORTANCE (top 30, final validation strategy) ===
geo_ts_mean         44.5292
geo_hour_mean       14.9390
road_type_filled    13.9772
road_type_ord       10.1854
lanes_x_road         5.1967
NumberofLanes        3.5138
rt_hour_mean         1.4223
large_veh_bin        1.3919
rt_ts_mean           1.1685
hour                 0.7103
geo_ts_delta         0.6667
ts_min               0.4458
geo5_freq            0.4347
sin_hour             0.2370
geo_freq             0.1872
geo4                 0.1663
weather_missing      0.1554
geo_mean             0.1527
geo_std              0.1194
geo4_freq            0.0899
rt_missing           0.0762
geo4_mean            0.0672
is_night             0.0546
cos_hour             0.0513
ts_mean              0.0388
temp_filled          0.0222
weather_filled       0.0002
temp_missing         0.0000
landmark_bin         0.0000
geo_count            0.0000

=== TARGETED IMPORTANCE REVIEW ===
            Feature  In Final Features  Importance
        ge

In [18]:
print('=== RANKED FEATURE ADDITIONS ===')
print(ranked_improvements.assign(
    **{
        'Validation R2 Before': ranked_improvements['Validation R2 Before'].round(6),
        'Validation R2 After': ranked_improvements['Validation R2 After'].round(6),
        'Gain': ranked_improvements['Gain'].round(6),
    }
).to_string(index=False))

print()
print('Features kept from v3 experiments:')
print(feature_eval_df.loc[feature_eval_df['Kept'], 'Experiment'].to_string(index=False))

=== RANKED FEATURE ADDITIONS ===
          Experiment  Validation R2 Before  Validation R2 After      Gain  Kept
seen_unseen_features              0.664431             0.664431  0.000000 False
    rt_ts_confidence              0.664431             0.585822 -0.078609 False
      count_features              0.664431             0.574533 -0.089898 False
 geo_hour_confidence              0.664431             0.517799 -0.146633 False
  smoothed_encodings              0.664431             0.455773 -0.208658 False
   geo_ts_confidence              0.664431             0.331209 -0.333222 False

Features kept from v3 experiments:
Series([], )


## 8. Train on Full Dataset

In [9]:
print('Building full-train features...')
full_fe = build_features(train, train)
y_full = full_fe['demand']


def full_iterations_from(model):
    best_tree_count = getattr(model, 'best_iteration_', None)
    if best_tree_count is None or best_tree_count <= 0:
        best_tree_count = getattr(model, 'tree_count_', 1000)
    return max(100, int(best_tree_count * 1.10))


def train_full_catboost(param_overrides, validation_model):
    full_params = CAT_BASE_PARAMS.copy()
    full_params.update(param_overrides)
    full_params.pop('early_stopping_rounds', None)
    full_params.pop('eval_metric', None)
    full_params['iterations'] = full_iterations_from(validation_model)

    model = CatBoostRegressor(
        **full_params,
        cat_features=[c for c in final_cat_features if c in final_features],
        verbose=100,
    )
    model.fit(full_fe[final_features], y_full)
    return model

if final_strategy == 'ensemble':
    print('Training full CatBoost ensemble models...')
    final_full_models = {}
    for model_name, weight in best_ensemble['Weights'].items():
        print(f'Training full model {model_name} with weight {weight:.2f}')
        final_full_models[model_name] = train_full_catboost(
            ensemble_specs[model_name],
            ensemble_models[model_name],
        )
else:
    print('Training full best CatBoost model...')
    final_full_model = train_full_catboost(best_cat_params, best_cat_model)

print('Full-train final model(s) trained.')

Building full-train features...
Training full best CatBoost model...
0:	learn: 0.1355618	total: 9.23ms	remaining: 914ms
99:	learn: 0.0166858	total: 567ms	remaining: 0us
Full-train final model(s) trained.


## 9. Predict Test & Generate Submission

In [10]:
print('Building test features...')
test_fe = build_features(test, train)

if final_strategy == 'ensemble':
    pred_parts = []
    for model_name, weight in best_ensemble['Weights'].items():
        pred_parts.append(weight * final_full_models[model_name].predict(test_fe[final_features]))
    preds_test = np.sum(pred_parts, axis=0)
else:
    preds_test = final_full_model.predict(test_fe[final_features])

preds_test = np.clip(preds_test, 0.0, 1.0)

print(f'Prediction stats: min={preds_test.min():.5f}  mean={preds_test.mean():.5f}  max={preds_test.max():.5f}')
print(f'Final strategy: {final_strategy}')
if final_strategy == 'ensemble':
    print('Final ensemble weights:', best_ensemble['Weights'])
else:
    print('Final CatBoost params:', best_cat_params)

Building test features...
Prediction stats: min=0.00572  mean=0.11266  max=0.99158
Final strategy: single_catboost
Final CatBoost params: {'depth': 6, 'l2_leaf_reg': 7, 'learning_rate': 0.05}


In [11]:
submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': preds_test,
})

submission.to_csv('submission.csv', index=False)

print('submission.csv written successfully.')
print(f'Shape: {submission.shape}')
print(submission.head(10).to_string())

print()
print('=== FINAL SUMMARY ===')
print(f'Best validation R2   : {best_validation_r2:.6f}')
print(f'Best CatBoost R2     : {best_cat_r2:.6f}')
print(f'Best CatBoost params : {best_cat_params}')
print(f'Best ensemble R2     : {best_ensemble_r2:.6f}')
print(f'Best ensemble params : {best_ensemble}')
print(f'Final strategy       : {final_strategy}')
print()
print('Kept v3 feature experiments:')
kept = feature_eval_df.loc[feature_eval_df['Kept'], 'Experiment']
print(kept.to_string(index=False) if len(kept) else 'None')

submission.csv written successfully.
Shape: (41778, 2)
   Index    demand
0      0  0.044175
1      1  0.030409
2      2  0.031216
3      3  0.070378
4      4  0.105532
5      5  0.014888
6      6  0.020585
7      7  0.248924
8      8  0.022302
9      9  0.072713

=== FINAL SUMMARY ===
Best validation R2   : 0.664431
Best CatBoost R2     : 0.664431
Best CatBoost params : {'depth': 6, 'l2_leaf_reg': 7, 'learning_rate': 0.05}
Best ensemble R2     : 0.664343
Best ensemble params : {'Blend': 'A+B', 'Weights': {'A': 0.05, 'B': 0.95}, 'Validation R2': 0.6643434429901064}
Final strategy       : single_catboost

Kept v3 feature experiments:
None


## 10. Summary

The executed cells above print:

| Item | Output |
|------|--------|
| Diagnostic report | Phase 1 diagnostics cell output |
| Count-feature results | Feature experiment results |
| Seen/unseen results | Feature experiment results |
| Smoothing results | Smoothing table |
| Hyperparameter results | Micro-tuning table |
| Ensemble results | Single-model and blend tables |
| Best validation R2 | Final summary cell |
| Final feature list | Final feature list table |
| Top feature importances | Top 30 table and targeted review |
| Validation design | Timestamp-aligned day-48 holdout, unchanged |
| Submission | `submission.csv` from the best temporal-validation strategy |